In [1]:
# Cell 1 — Environment and project inventory
# Read-only: no downloads, training, or changes to existing files.

from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import csv
import platform
import sys

# If automatic detection fails, enter your project folder here.
# Example: PROJECT_ROOT_OVERRIDE = r"C:\...\From-Soundscapes-to-Biodiversity"
PROJECT_ROOT_OVERRIDE = None

REQUIRED_FILES = {
    "Recording features": "data/processed/acoustic_features_recording.csv",
    "Window features": "data/processed/acoustic_features_window.csv",
    "Recording metadata": "data/processed/master_metadata.csv",
    "Feature manifest": "data/manifests/feature_manifest.csv",
}

print("ENVIRONMENT")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

print("\nPACKAGE VERSIONS")
for package in [
    "numpy", "pandas", "scipy", "scikit-learn",
    "matplotlib", "seaborn", "librosa", "scikit-maad",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# Search only nearby, explicitly defined paths—not your whole computer.
if PROJECT_ROOT_OVERRIDE:
    candidates = [Path(PROJECT_ROOT_OVERRIDE).expanduser()]
else:
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([
            parent,
            parent / "work" / "project",
            parent / "From-Soundscapes-to-Biodiversity",
        ])

matches = []
for candidate in candidates:
    if all((candidate / rel).is_file() for rel in REQUIRED_FILES.values()):
        resolved = candidate.resolve()
        if resolved not in matches:
            matches.append(resolved)

PROJECT_ROOT = None

if len(matches) == 1:
    PROJECT_ROOT = matches[0]
    print("\nPROJECT ROOT:", PROJECT_ROOT)

    print("\nCSV INVENTORY")
    for label, relative_path in REQUIRED_FILES.items():
        path = PROJECT_ROOT / relative_path

        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle)
            columns = next(reader)
            row_count = sum(1 for row in reader if row)

        print(f"\n{label}")
        print("  File:", relative_path)
        print("  Rows:", row_count)
        print("  Columns:", len(columns))
        print("  First columns:", ", ".join(columns[:8]))

    audio_dir = PROJECT_ROOT / "audio"
    audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    audio_count = (
        sum(
            1 for path in audio_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in audio_extensions
        )
        if audio_dir.exists() else 0
    )

    print("\nRESEARCH AUDIO")
    print("Files under project/audio:", audio_count)
    print("This does not check any separate Drive or external archive.")
    print("\nCheckpoint complete: inventory only; data not yet validated.")

elif not matches:
    print("\nPROJECT NOT FOUND.")
    print("Set PROJECT_ROOT_OVERRIDE to the folder containing data/ and research/.")
    print("If using Colab, your existing Drive project must be accessible first.")

else:
    print("\nMULTIPLE PROJECT COPIES FOUND:")
    for match in matches:
        print(" ", match)
    print("Set PROJECT_ROOT_OVERRIDE to the copy you intend to use.")

ENVIRONMENT
Python: 3.11.9
Platform: Windows-10-10.0.26200-SP0
Working directory: c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks

PACKAGE VERSIONS
numpy: 2.4.6
pandas: 2.2.2
scipy: 1.17.1
scikit-learn: 1.9.1
matplotlib: 3.11.2
seaborn: 0.13.2
librosa: 0.10.2
scikit-maad: 1.5.1

PROJECT ROOT: C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

CSV INVENTORY

Recording features
  File: data/processed/acoustic_features_recording.csv
  Rows: 275
  Columns: 114
  First columns: recording_id, state, species_id, common_name, scientific_name, original_sample_rate, original_channels, flag_short_duration

Window features
  File: data/processed/acoustic_features_window.csv
  Rows: 1179
  Columns: 69
  First columns: window_duration_seconds, aci, adi, bioacoustic_index, ndsi, ndsi_ratio_ba, anthropophony, biophony

Recording metadata
  File: data/processed/master_metadata.csv
  Rows: 275
  Columns: 41
  First columns: state, recording_id, common_name, s

In [2]:
# Cell 2 — Identity, label, window, and feature-manifest integrity
# No cleaning, exclusions, transformations, or modeling yet.

import pandas as pd

assert PROJECT_ROOT is not None, "Run Cell 1 and resolve the project path first."

recordings = pd.read_csv(
    PROJECT_ROOT / REQUIRED_FILES["Recording features"],
    dtype={"recording_id": "string"},
)
windows = pd.read_csv(
    PROJECT_ROOT / REQUIRED_FILES["Window features"],
    dtype={"recording_id": "string"},
)
metadata = pd.read_csv(
    PROJECT_ROOT / REQUIRED_FILES["Recording metadata"],
    dtype={"recording_id": "string"},
)
manifest = pd.read_csv(
    PROJECT_ROOT / REQUIRED_FILES["Feature manifest"],
    dtype="string",
)

issues = []

def check(description, passed, detail=""):
    passed = bool(passed)
    print(f"{'PASS' if passed else 'CHECK'} | {description}")
    if detail:
        print("      ", detail)
    if not passed:
        issues.append(description)

print("RECORDING IDENTITIES")

for name, frame in [
    ("Recording features", recordings),
    ("Metadata", metadata),
    ("Windows", windows),
]:
    ids = frame["recording_id"]
    check(f"{name}: no missing IDs", ids.notna().all())
    check(
        f"{name}: no blank/padded IDs",
        ids.dropna().str.strip().ne("").all()
        and ids.dropna().eq(ids.dropna().str.strip()).all(),
    )
    if name != "Windows":
        check(f"{name}: unique IDs", not ids.duplicated().any())

recording_ids = set(recordings["recording_id"].dropna())
metadata_ids = set(metadata["recording_id"].dropna())
window_ids = set(windows["recording_id"].dropna())

check(
    "Recording and metadata ID sets agree",
    recording_ids == metadata_ids,
    f"Features-only: {len(recording_ids - metadata_ids)}; "
    f"metadata-only: {len(metadata_ids - recording_ids)}",
)
check(
    "Window and recording ID sets agree",
    window_ids == recording_ids,
    f"Window-only: {len(window_ids - recording_ids)}; "
    f"recordings without windows: {len(recording_ids - window_ids)}",
)

# Only perform joins when keys are safe.
safe_keys = all(
    frame["recording_id"].notna().all()
    and frame["recording_id"].is_unique
    for frame in [recordings, metadata]
)

print("\nLABEL CONSISTENCY")

labels = ["state", "species_id", "scientific_name"]

for name, frame in [
    ("Recording features", recordings),
    ("Metadata", metadata),
    ("Windows", windows),
]:
    for label in labels:
        check(
            f"{name}: {label} present and nonblank",
            frame[label].notna().all()
            and frame[label].astype("string").str.strip().ne("").all(),
        )
    check(
        f"{name}: state labels limited to CA/AZ/TX",
        frame["state"].isin(["CA", "AZ", "TX"]).all(),
    )

if safe_keys:
    label_join = recordings[["recording_id", *labels]].merge(
        metadata[["recording_id", *labels]],
        on="recording_id",
        how="inner",
        validate="one_to_one",
        suffixes=("_features", "_metadata"),
    )

    for label in labels:
        agrees = label_join[f"{label}_features"].eq(
            label_join[f"{label}_metadata"]
        ).fillna(False)
        check(
            f"Features versus metadata: {label} agrees",
            agrees.all(),
            f"Mismatched joined rows: {(~agrees).sum()}",
        )
        if not agrees.all():
            print(label_join.loc[~agrees].head(5).to_string(index=False))

    window_labels = windows[["recording_id", *labels]].merge(
        recordings[["recording_id", *labels]],
        on="recording_id",
        how="left",
        validate="many_to_one",
        suffixes=("_window", "_recording"),
    )

    for label in labels:
        agrees = window_labels[f"{label}_window"].eq(
            window_labels[f"{label}_recording"]
        ).fillna(False)
        check(
            f"Windows versus parent recording: {label} agrees",
            agrees.all(),
            f"Mismatched window rows: {(~agrees).sum()}",
        )
else:
    print("Label joins skipped: resolve missing/duplicate keys first.")

print("\nWINDOW MEMBERSHIP")

check(
    "Unique recording_id + window_index pairs",
    not windows.duplicated(["recording_id", "window_index"]).any(),
)
check("No missing window_index", windows["window_index"].notna().all())

if recordings["recording_id"].is_unique:
    observed_counts = windows.groupby("recording_id").size()
    count_check = recordings[["recording_id", "n_windows"]].copy()
    count_check["observed_windows"] = (
        count_check["recording_id"].map(observed_counts).fillna(0).astype(int)
    )
    expected_counts = pd.to_numeric(
        count_check["n_windows"], errors="coerce"
    )
    agrees = expected_counts.eq(count_check["observed_windows"])
    check("Reported n_windows matches actual rows", agrees.all())
    if not agrees.all():
        print(count_check.loc[~agrees].head(10).to_string(index=False))

print("\nFEATURE MANIFEST")

check("No missing feature names", manifest["feature_name"].notna().all())
check("Unique manifest feature names", manifest["feature_name"].is_unique)

missing_features = sorted(
    set(manifest["feature_name"].dropna()) - set(recordings.columns)
)
check(
    "All manifest features exist in the recording table",
    not missing_features,
    f"Missing: {missing_features}",
)

print("\nManifest aggregation counts:")
print(manifest["aggregation"].value_counts(dropna=False).to_string())

mean_features = manifest.loc[
    manifest["aggregation"].eq("mean_across_windows"),
    "feature_name",
].tolist()

check("Initial mean-feature list contains 50 entries", len(mean_features) == 50)

print("\nMean features by manifest family:")
print(
    manifest.loc[
        manifest["feature_name"].isin(mean_features), "feature_group"
    ].value_counts(dropna=False).to_string()
)

print("\nSTATE SUPPORT — recording rows")
print(recordings["state"].value_counts(dropna=False).to_string())

print("\nCHECKPOINT SUMMARY")
if issues:
    print(f"{len(issues)} checks need attention:")
    for issue in issues:
        print(" -", issue)
else:
    print("All identity, label, membership, and manifest checks passed.")

print("No rows removed. No values changed. No models fitted.")

RECORDING IDENTITIES
PASS | Recording features: no missing IDs
PASS | Recording features: no blank/padded IDs
PASS | Recording features: unique IDs
PASS | Metadata: no missing IDs
PASS | Metadata: no blank/padded IDs
PASS | Metadata: unique IDs
PASS | Windows: no missing IDs
PASS | Windows: no blank/padded IDs
PASS | Recording and metadata ID sets agree
       Features-only: 0; metadata-only: 0
PASS | Window and recording ID sets agree
       Window-only: 0; recordings without windows: 0

LABEL CONSISTENCY
PASS | Recording features: state present and nonblank
PASS | Recording features: species_id present and nonblank
PASS | Recording features: scientific_name present and nonblank
PASS | Recording features: state labels limited to CA/AZ/TX
PASS | Metadata: state present and nonblank
PASS | Metadata: species_id present and nonblank
PASS | Metadata: scientific_name present and nonblank
PASS | Metadata: state labels limited to CA/AZ/TX
PASS | Windows: state present and nonblank
PASS | Wind

In [3]:
# Cell 3 — Numerical quality, recording conditions, and audio inventory
# Read-only. No exclusions, imputation, transformations, or modeling.

import numpy as np
import pandas as pd

# ---------- A. Numerical feature audit ----------

raw_features = recordings[mean_features]
numeric_features = raw_features.apply(pd.to_numeric, errors="coerce")

# Distinguish original missing values from failed numeric conversion.
conversion_failures = raw_features.notna() & numeric_features.isna()
feature_array = numeric_features.to_numpy(dtype=float)

# Replace infinities only in a temporary descriptive view.
# The original data and numeric_features remain unchanged.
finite_view = numeric_features.replace([np.inf, -np.inf], np.nan)

feature_audit = pd.DataFrame({
    "feature": mean_features,
    "missing": raw_features.isna().sum().to_numpy(),
    "conversion_failures": conversion_failures.sum().to_numpy(),
    "infinite": np.isinf(feature_array).sum(axis=0),
    "negative": (numeric_features < 0).sum().to_numpy(),
    "zero": (numeric_features == 0).sum().to_numpy(),
    "unique_finite": finite_view.nunique().to_numpy(),
    "minimum": finite_view.min().to_numpy(),
    "median": finite_view.median().to_numpy(),
    "maximum": finite_view.max().to_numpy(),
})

print("A. NUMERICAL FEATURE AUDIT")
print("Mean-feature matrix shape:", numeric_features.shape)
print("Missing cells:", int(feature_audit["missing"].sum()))
print("Failed numeric conversions:",
      int(feature_audit["conversion_failures"].sum()))
print("Infinite cells:", int(feature_audit["infinite"].sum()))
print("Features containing negative values:",
      int((feature_audit["negative"] > 0).sum()))

problem_features = feature_audit.loc[
    (feature_audit["missing"] > 0)
    | (feature_audit["conversion_failures"] > 0)
    | (feature_audit["infinite"] > 0)
    | (feature_audit["unique_finite"] <= 1)
]

print("\nMissing/non-numeric/infinite/constant features:")
print(
    problem_features.to_string(index=False)
    if not problem_features.empty else "None"
)

print("\nFeatures containing negative values:")
print(
    feature_audit.loc[
        feature_audit["negative"] > 0,
        ["feature", "negative", "minimum", "median", "maximum"],
    ].to_string(index=False)
)

bad_numeric_rows = (
    numeric_features.isna().any(axis=1)
    | pd.Series(np.isinf(feature_array).any(axis=1),
                index=recordings.index)
)
print("\nRecordings with at least one invalid/missing feature:",
      int(bad_numeric_rows.sum()))

# ---------- B. Duration and window coverage ----------

conditions = recordings[
    ["recording_id", "state", "n_windows",
     "original_sample_rate", "original_channels"]
].merge(
    metadata[["recording_id", "duration_seconds", "quality"]],
    on="recording_id",
    how="left",
    validate="one_to_one",
)

conditions["duration_seconds"] = pd.to_numeric(
    conditions["duration_seconds"], errors="coerce"
)

print("\nB. RECORDING CONDITIONS")
print("\nDuration in seconds, by state:")
print(
    conditions.groupby("state")["duration_seconds"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(3).to_string()
)

print("\nWindow counts, by state:")
print(
    conditions.groupby("state")["n_windows"]
    .agg(["min", "median", "max", "sum"]).to_string()
)

short_recordings = conditions.loc[
    conditions["duration_seconds"] < 3,
    ["recording_id", "state", "duration_seconds", "n_windows", "quality"],
]
print("\nRecordings shorter than 3 seconds — inspection only:")
print(
    short_recordings.sort_values("duration_seconds").to_string(index=False)
    if not short_recordings.empty else "None"
)

print("\nOriginal sample rate × state:")
print(pd.crosstab(
    conditions["original_sample_rate"].fillna("MISSING"),
    conditions["state"],
).to_string())

print("\nQuality × state:")
print(pd.crosstab(
    conditions["quality"].fillna("MISSING"),
    conditions["state"],
).to_string())

# ---------- C. Quality flags ----------
# Avoid astype(bool): the string "False" would incorrectly become True.

print("\nC. RECORDING FLAGS")
flag_columns = [c for c in recordings.columns if c.startswith("flag_")]

flag_rows = []
for column in flag_columns:
    values = recordings[column].astype("string").str.strip().str.lower()
    flag_rows.append({
        "flag": column,
        "true": int(values.isin(["true", "1", "1.0"]).sum()),
        "false": int(values.isin(["false", "0", "0.0"]).sum()),
        "missing": int(values.isna().sum()),
        "unrecognized": int((
            values.notna()
            & ~values.isin(["true", "false", "1", "0", "1.0", "0.0"])
        ).sum()),
    })

print(pd.DataFrame(flag_rows).to_string(index=False))

# ---------- D. Audio directory inventory ----------
# List files only: no decoding, hashing, renaming, or deletion.

audio_dir = PROJECT_ROOT / "audio"
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

audio_rows = []
if audio_dir.exists():
    for path in sorted(audio_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in audio_extensions:
            relative = path.relative_to(audio_dir)
            audio_rows.append({
                "folder": str(relative.parent),
                "extension": path.suffix.lower(),
                "filename": path.name,
                "bytes": path.stat().st_size,
            })

audio_inventory = pd.DataFrame(
    audio_rows, columns=["folder", "extension", "filename", "bytes"]
)

print("\nD. AUDIO FILE ORGANIZATION")
if audio_inventory.empty:
    print("No audio files found under project/audio.")
else:
    audio_summary = (
        audio_inventory.groupby(["folder", "extension"])
        .agg(files=("filename", "size"), total_bytes=("bytes", "sum"))
        .reset_index()
    )
    audio_summary["total_MiB"] = (
        audio_summary.pop("total_bytes") / (1024 ** 2)
    ).round(2)
    print(audio_summary.to_string(index=False))

    print("\nExample filenames — up to three per folder/extension:")
    print(
        audio_inventory.groupby(["folder", "extension"], group_keys=False)
        .head(3)[["folder", "extension", "filename"]]
        .to_string(index=False)
    )

print("\nNo data changed. Duration thresholds above are diagnostic, not exclusions.")

A. NUMERICAL FEATURE AUDIT
Mean-feature matrix shape: (275, 50)
Missing cells: 0
Failed numeric conversions: 0
Infinite cells: 0
Features containing negative values: 14

Missing/non-numeric/infinite/constant features:
None

Features containing negative values:
     feature  negative     minimum      median    maximum
   ndsi_mean       102   -0.999595    0.197913   0.999949
mfcc_01_mean       275 -415.778748 -203.189941 -37.796223
mfcc_02_mean        21  -59.767708   29.802359 108.311186
mfcc_03_mean       181  -66.236839   -4.555608  29.943426
mfcc_04_mean        61  -28.154297    4.900884  31.889248
mfcc_05_mean       161  -22.240019   -0.883212  15.152509
mfcc_06_mean       131  -27.870883    0.222978  22.525702
mfcc_07_mean       152  -27.776726   -0.558575  32.014882
mfcc_08_mean       125  -14.942760    0.305139  16.629961
mfcc_09_mean       151  -18.441050   -0.205596  12.582631
mfcc_10_mean        92  -20.035595    0.644782  15.492499
mfcc_11_mean       114  -13.384805    0.225

In [4]:
# Cell 4 — Audio membership and processing provenance
# Read-only. Do not infer that equally sized folders contain identical IDs.

import json
import re
import pandas as pd

expected_ids = set(recordings["recording_id"])
expected_states = recordings.set_index("recording_id")["state"].to_dict()

audio_membership = {}
pattern = re.compile(r"^(CA|AZ|TX)_(\d+)$", flags=re.IGNORECASE)

print("A. AUDIO MEMBERSHIP")

for folder_name in ["raw", "validated"]:
    folder = PROJECT_ROOT / "audio" / folder_name
    rows = []
    unparsed = []

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in audio_extensions:
                continue

            match = pattern.fullmatch(path.stem)
            if match:
                rows.append({
                    "recording_id": match.group(2),
                    "filename_state": match.group(1).upper(),
                    "relative_path": str(path.relative_to(PROJECT_ROOT)),
                    "bytes": path.stat().st_size,
                })
            else:
                unparsed.append(str(path.relative_to(PROJECT_ROOT)))

    table = pd.DataFrame(
        rows,
        columns=["recording_id", "filename_state", "relative_path", "bytes"],
    )
    audio_membership[folder_name] = table
    folder_ids = set(table["recording_id"])

    missing = sorted(expected_ids - folder_ids)
    extra = sorted(folder_ids - expected_ids)
    duplicate_rows = table.loc[
        table["recording_id"].duplicated(keep=False)
    ]

    matched = table.loc[table["recording_id"].isin(expected_ids)].copy()
    matched["expected_state"] = matched["recording_id"].map(expected_states)
    wrong_state = matched.loc[
        matched["filename_state"] != matched["expected_state"]
    ]

    print(f"\n{folder_name.upper()}")
    print("Parsed files:", len(table))
    print("Unique recording IDs:", len(folder_ids))
    print("Expected IDs present:", len(folder_ids & expected_ids))
    print("Missing expected IDs:", len(missing))
    print("Extra IDs outside feature table:", len(extra))
    print("Duplicate-ID file rows:", len(duplicate_rows))
    print("Filename-state disagreements:", len(wrong_state))
    print("Unparsed filenames:", len(unparsed))
    print("Zero-byte files:", int(table["bytes"].eq(0).sum()))

    if missing:
        print("Missing IDs:", missing)
    if extra:
        print("Extra IDs:", extra)
    if not duplicate_rows.empty:
        print(duplicate_rows.to_string(index=False))
    if not wrong_state.empty:
        print(wrong_state.to_string(index=False))
    if unparsed:
        print("Unparsed examples:", unparsed[:10])

raw_ids = set(audio_membership["raw"]["recording_id"])
validated_ids = set(audio_membership["validated"]["recording_id"])

print("\nRAW VERSUS VALIDATED")
print("Shared IDs:", len(raw_ids & validated_ids))
print("Raw only:", len(raw_ids - validated_ids))
print("Validated only:", len(validated_ids - raw_ids))

# Show acquisition/cleaning records without assuming their schema.
print("\nB. ACQUISITION AND CLEANING RECORDS")

provenance_files = [
    "data/manifests/a1_xenocanto_selection_manifest.csv",
    "data/manifests/cleaning_log.csv",
    "data/processed/cleaning_summary.csv",
]

for relative in provenance_files:
    path = PROJECT_ROOT / relative
    print(f"\n{relative}")

    if not path.exists():
        print("NOT FOUND")
        continue

    frame = pd.read_csv(path, dtype="string")
    print("Shape:", frame.shape)
    print("Columns:", ", ".join(frame.columns))

    # Avoid printing long URLs or arbitrary free-text fields.
    useful = [
        c for c in frame.columns
        if any(term in c.lower() for term in [
            "recording_id", "state", "action", "reason",
            "status", "count", "selection_method",
        ])
    ]
    if useful:
        print(frame[useful].head(8).to_string(index=False))

    if "recording_id" in frame.columns:
        ids = set(frame["recording_id"].dropna())
        print("IDs overlapping feature table:", len(ids & expected_ids))
        print("IDs outside feature table:", len(ids - expected_ids))

# Inspect relevant source lines, not stored notebook outputs.
print("\nC. NOTEBOOK PROCESSING REFERENCES")

reference_pattern = re.compile(
    r"AUDIO_DIR|RAW_AUDIO|VALIDATED|validated|"
    r"FEATURE_CONFIG|analysis_sample_rate|"
    r"short_recording_policy|window_seconds|window_overlap|"
    r"primary_frequency|flag_bandlimit|"
    r"flag_low_amplitude|flag_clipping|flag_short_duration|"
    r"to_csv|acoustic_features_recording",
    flags=re.IGNORECASE,
)

for notebook_name in ["04_cleaning.ipynb", "05_audio_features.ipynb"]:
    path = PROJECT_ROOT / "research" / "notebooks" / notebook_name
    print(f"\n{notebook_name}")

    if not path.exists():
        print("NOT FOUND")
        continue

    notebook = json.loads(path.read_text(encoding="utf-8-sig"))
    references = []

    for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        source = source if isinstance(source, str) else "".join(source)

        for line in source.splitlines():
            if reference_pattern.search(line):
                references.append(
                    f"Cell {cell_number}: {line.strip()[:220]}"
                )

    print("\n".join(references[:65]) or "No matching references found.")
    if len(references) > 65:
        print(f"... {len(references) - 65} further matching lines omitted.")

print("\nNo files modified, deleted, decoded, or downloaded.")

A. AUDIO MEMBERSHIP

RAW
Parsed files: 275
Unique recording IDs: 274
Expected IDs present: 42
Missing expected IDs: 233
Extra IDs outside feature table: 232
Duplicate-ID file rows: 2
Filename-state disagreements: 1
Unparsed filenames: 0
Zero-byte files: 0
Missing IDs: ['1035206', '1080401', '109164', '109171', '1106496', '110689', '114789', '1152009', '120845', '123297', '123582', '130831', '130846', '132208', '132209', '132328', '132452', '132678', '132684', '132844', '132890', '132942', '132946', '132948', '132967', '132968', '133346', '133556', '134480', '135420', '135438', '1382', '140011', '140797', '141468', '145955', '145957', '147270', '147312', '150244', '154199', '154976', '154983', '163986', '167459', '168805', '172887', '175369', '17795', '179040', '193254', '193255', '195342', '196870', '196879', '200277', '210719', '212892', '216259', '216516', '219022', '219976', '222769', '223349', '224822', '228428', '228580', '229133', '230019', '230023', '231530', '231963', '232953',

In [5]:
#  — Acquisition consistency and exact-file duplicate audit
# Read-only. SHA-256 detects identical files, not all duplicate recordings.

import hashlib
import pandas as pd

selection = pd.read_csv(
    PROJECT_ROOT / "data/manifests/a1_xenocanto_selection_manifest.csv",
    dtype={"recording_id": "string"},
)

print("A. ACQUISITION MANIFEST")

selection_keys_ok = (
    selection["recording_id"].notna().all()
    and selection["recording_id"].is_unique
)

print("Nonmissing, unique IDs:", selection_keys_ok)
print(
    "Exact match to feature-table IDs:",
    set(selection["recording_id"].dropna()) == expected_ids,
)

if selection_keys_ok:
    comparison = recordings[
        ["recording_id", "state", "scientific_name"]
    ].merge(
        selection[["recording_id", "state", "scientific_name"]],
        on="recording_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_features", "_selection"),
        indicator=True,
    )

    print("Join membership:")
    print(comparison["_merge"].value_counts().to_string())

    for label in ["state", "scientific_name"]:
        same = comparison[f"{label}_features"].eq(
            comparison[f"{label}_selection"]
        ).fillna(False)

        print(f"{label} disagreements:", int((~same).sum()))
        if not same.all():
            print(
                comparison.loc[
                    ~same,
                    ["recording_id",
                     f"{label}_features", f"{label}_selection"],
                ].head(10).to_string(index=False)
            )
else:
    print("Manifest comparison skipped: resolve key problems first.")

for column in ["selection_method", "selection_seed"]:
    if column in selection.columns:
        print(f"\n{column}:")
        print(selection[column].value_counts(dropna=False).to_string())

print("\nB. VALIDATED AUDIO HASHES")

validated_files = audio_membership["validated"].copy()

assert validated_files["recording_id"].is_unique
assert set(validated_files["recording_id"]) == expected_ids

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hash_rows = []

for number, row in enumerate(
    validated_files.itertuples(index=False), start=1
):
    path = PROJECT_ROOT / row.relative_path

    try:
        digest = sha256_file(path)
        error = None
    except OSError as exc:
        digest = None
        error = str(exc)

    hash_rows.append({
        "recording_id": row.recording_id,
        "state": row.filename_state,
        "relative_path": row.relative_path,
        "sha256": digest,
        "read_error": error,
    })

    if number % 50 == 0 or number == len(validated_files):
        print(f"Checked {number}/{len(validated_files)} files")

audio_hash_audit = pd.DataFrame(hash_rows)
read_failures = audio_hash_audit.loc[
    audio_hash_audit["read_error"].notna()
]

successfully_read = audio_hash_audit.loc[
    audio_hash_audit["sha256"].notna()
]
identical_files = successfully_read.loc[
    successfully_read["sha256"].duplicated(keep=False)
].sort_values(["sha256", "recording_id"])

print("\nRead failures:", len(read_failures))
print("Distinct file hashes:", successfully_read["sha256"].nunique())
print("Files belonging to exact-duplicate groups:", len(identical_files))

if not read_failures.empty:
    print(read_failures[["recording_id", "read_error"]].to_string(index=False))

if not identical_files.empty:
    print(
        identical_files[
            ["recording_id", "state", "relative_path", "sha256"]
        ].to_string(index=False)
    )
else:
    print("No byte-identical files detected among successfully read files.")

print(
    "\nInterpretation limit: different hashes do not rule out re-encoded,"
    " overlapping, or same-encounter recordings."
)
print("Audit tables remain in notebook memory. No files changed.")

A. ACQUISITION MANIFEST
Nonmissing, unique IDs: True
Exact match to feature-table IDs: True
Join membership:
_merge
both          275
left_only       0
right_only      0
state disagreements: 0
scientific_name disagreements: 0

selection_method:
selection_method
species_stratified_round_robin    275

selection_seed:
selection_seed
42    275

B. VALIDATED AUDIO HASHES
Checked 50/275 files
Checked 100/275 files
Checked 150/275 files
Checked 200/275 files
Checked 250/275 files
Checked 275/275 files

Read failures: 0
Distinct file hashes: 275
Files belonging to exact-duplicate groups: 0
No byte-identical files detected among successfully read files.

Interpretation limit: different hashes do not rule out re-encoded, overlapping, or same-encounter recordings.
Audit tables remain in notebook memory. No files changed.


In [6]:
#  Sampling structure and potential dependency groups
# Read-only. Coordinate pairs and date/location combinations are proxies,
# not yet validated sites or independent recording encounters.


STATES = ["CA", "AZ", "TX"]

sampling = metadata.copy()

# ---------- A. Species coverage ----------

species_state_counts = pd.crosstab(
    sampling["scientific_name"], sampling["state"]
).reindex(columns=STATES, fill_value=0)

species_coverage = species_state_counts.copy()
species_coverage["total_recordings"] = species_state_counts.sum(axis=1)
species_coverage["states_present"] = species_state_counts.gt(0).sum(axis=1)

print("A. SPECIES COVERAGE")
print("Total species:", len(species_coverage))
print(
    "Species present in exactly 1 / 2 / 3 states:",
    species_coverage["states_present"]
    .value_counts().reindex([1, 2, 3], fill_value=0).to_dict(),
)
print(
    "Species with only one recording:",
    int(species_coverage["total_recordings"].eq(1).sum()),
)

print("\nSpecies occurring in all three states:")
print(
    species_coverage.loc[
        species_coverage["states_present"].eq(3)
    ].sort_values("total_recordings", ascending=False).to_string()
)

print("\nPairwise species overlap:")
for a, b in [("CA", "AZ"), ("CA", "TX"), ("AZ", "TX")]:
    shared = species_state_counts[a].gt(0) & species_state_counts[b].gt(0)
    replicated = (
        species_state_counts[a].ge(2)
        & species_state_counts[b].ge(2)
    )
    print(
        f"{a}–{b}: {shared.sum()} shared species; "
        f"{replicated.sum()} have ≥2 recordings in BOTH states"
    )

print("\nRecordings from species found only in their sampled state:")
single_state_species = set(
    species_coverage.index[species_coverage["states_present"].eq(1)]
)
sampling["sample_state_exclusive_species"] = (
    sampling["scientific_name"].isin(single_state_species)
)
print(
    sampling.groupby("state")["sample_state_exclusive_species"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "recordings", "mean": "fraction"})
    .reindex(STATES).round(3).to_string()
)
print("These are sample-specific occurrences, not species range claims.")

# ---------- B. Coordinates ----------

latitude = pd.to_numeric(sampling["latitude"], errors="coerce")
longitude = pd.to_numeric(sampling["longitude"], errors="coerce")

valid_coordinates = (
    latitude.between(-90, 90)
    & longitude.between(-180, 180)
)
sampling["coordinate_valid_for_audit"] = valid_coordinates

print("\nB. COORDINATE COVERAGE")
print("Missing/invalid coordinates:", int((~valid_coordinates).sum()))

# Numeric tuple keys avoid treating formatting differences as new locations.
sampling["coordinate_key"] = pd.Series(
    [
        (float(lat), float(lon)) if valid else None
        for lat, lon, valid in zip(latitude, longitude, valid_coordinates)
    ],
    index=sampling.index,
    dtype="object",
)

coordinate_groups = (
    sampling.loc[valid_coordinates]
    .groupby("coordinate_key")
    .agg(
        recordings=("recording_id", "size"),
        species=("scientific_name", "nunique"),
        state_count=("state", "nunique"),
        states=("state", lambda x: ",".join(sorted(set(x)))),
    )
    .sort_values("recordings", ascending=False)
)

print("Exact numeric coordinate groups:", len(coordinate_groups))
print(
    "Groups with multiple recordings:",
    int(coordinate_groups["recordings"].gt(1).sum()),
)
print(
    "Coordinate groups spanning multiple state labels:",
    int(coordinate_groups["state_count"].gt(1).sum()),
)
print("\nTen largest coordinate groups:")
print(coordinate_groups.head(10).to_string())

print("\nCoordinate support by state:")
print(
    sampling.loc[valid_coordinates].groupby("state")
    .agg(
        recordings=("recording_id", "size"),
        coordinate_groups=("coordinate_key", "nunique"),
    ).reindex(STATES).to_string()
)

# ---------- C. Approximate recording occasions ----------

# Use an available date field, without inventing missing dates.
date_column = next(
    (c for c in ["date", "date_original", "datetime_local"]
     if c in sampling.columns),
    None,
)

print("\nC. LOCATION + DATE PROXY")

if date_column:
    sampling["audit_date"] = pd.to_datetime(
        sampling[date_column], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    usable = valid_coordinates & sampling["audit_date"].notna()
    occasion_groups = (
        sampling.loc[usable]
        .groupby(["coordinate_key", "audit_date"])
        .agg(
            recordings=("recording_id", "size"),
            species=("scientific_name", "nunique"),
            states=("state", lambda x: ",".join(sorted(set(x)))),
        )
        .sort_values("recordings", ascending=False)
    )

    print("Date field used:", date_column)
    print("Rows without usable location + date:", int((~usable).sum()))
    print(
        "Location/date groups with multiple recordings:",
        int(occasion_groups["recordings"].gt(1).sum()),
    )
    print("\nLargest repeated location/date groups:")
    print(
        occasion_groups.loc[
            occasion_groups["recordings"].gt(1)
        ].head(10).to_string()
    )
else:
    print("No suitable date field found.")

# ---------- D. Available provenance fields ----------

print("\nD. POTENTIAL SITE / RECORDIST FIELDS")
candidate_fields = [
    c for c in metadata.columns
    if any(term in c.lower() for term in [
        "recordist", "observer", "location", "site", "author",
    ])
]
print(candidate_fields or "None in master_metadata.")

print("\nNo groups finalized, recordings excluded, or splits created.")

A. SPECIES COVERAGE
Total species: 159
Species present in exactly 1 / 2 / 3 states: {1: 118, 2: 31, 3: 10}
Species with only one recording: 103

Species occurring in all three states:
state                     CA  AZ  TX  total_recordings  states_present
scientific_name                                                       
Anas carolinensis          2   2   1                 5               3
Archilochus alexandri      2   2   1                 5               3
Phalaenoptilus nuttallii   2   2   1                 5               3
Chordeiles minor           2   2   1                 5               3
Mareca strepera            2   2   1                 5               3
Geococcyx californianus    2   2   1                 5               3
Selasphorus platycercus    2   2   1                 5               3
Anas platyrhynchos         1   2   1                 4               3
Chordeiles acutipennis     1   2   1                 4               3
Antrostomus arizonae       1   2   

In [7]:
#  Recover original recording-location evidence


print("A. RECORDINGS AT THE QUESTIONABLE COORDINATE")

lat = pd.to_numeric(metadata["latitude"], errors="coerce")
lon = pd.to_numeric(metadata["longitude"], errors="coerce")

suspect_mask = (
    np.isclose(lat, 35.442, atol=0.00001, rtol=0)
    & np.isclose(lon, -94.9943, atol=0.00001, rtol=0)
)

suspect_records = metadata.loc[suspect_mask].copy()
suspect_ids = set(suspect_records["recording_id"])

display_columns = [
    c for c in [
        "recording_id", "state", "scientific_name",
        "latitude", "longitude", "source_url",
    ]
    if c in metadata.columns
]
print(suspect_records[display_columns].to_string(index=False))

print("\nB. CACHED XENO-CANTO METADATA")

# Search only project/data for likely source-metadata JSON files.
cache_paths = sorted(
    path for path in (PROJECT_ROOT / "data").rglob("*.json")
    if any(
        term in path.name.lower()
        for term in ["xc_metadata", "xenocanto", "xeno_canto"]
    )
)

print("Candidate cache files:", len(cache_paths))
for path in cache_paths[:20]:
    print(" ", path.relative_to(PROJECT_ROOT))
if len(cache_paths) > 20:
    print(" ... further filenames omitted")

def source_records(node):
    """Find recording objects in list or API-response JSON structures."""
    if isinstance(node, dict):
        if "id" in node and any(
            key in node for key in ["loc", "lat", "lng", "rec"]
        ):
            yield node
        else:
            for value in node.values():
                if isinstance(value, (dict, list)):
                    yield from source_records(value)
    elif isinstance(node, list):
        for value in node:
            yield from source_records(value)

source_rows = []
cache_errors = []

for path in cache_paths:
    try:
        payload = json.loads(path.read_text(encoding="utf-8-sig"))
        for item in source_records(payload):
            recording_id = str(item["id"]).strip()
            if recording_id not in expected_ids:
                continue

            source_rows.append({
                "recording_id": recording_id,
                "source_location": item.get("loc"),
                "source_country": item.get("cnt"),
                "source_latitude": item.get("lat"),
                "source_longitude": item.get("lng"),
                "recordist": item.get("rec"),
                "source_file": str(path.relative_to(PROJECT_ROOT)),
            })
    except (OSError, ValueError) as exc:
        cache_errors.append({
            "file": str(path.relative_to(PROJECT_ROOT)),
            "error": str(exc),
        })

source_evidence = pd.DataFrame(
    source_rows,
    columns=[
        "recording_id", "source_location", "source_country",
        "source_latitude", "source_longitude", "recordist", "source_file",
    ],
)

print("\nCurrent cohort IDs recovered:",
      source_evidence["recording_id"].nunique())
print("Current cohort IDs without cached evidence:",
      len(expected_ids - set(source_evidence["recording_id"])))

if cache_errors:
    print("\nCache read errors:")
    print(pd.DataFrame(cache_errors).to_string(index=False))

print("\nC. ORIGINAL LOCATION EVIDENCE FOR FLAGGED RECORDINGS")

suspect_evidence = source_evidence.loc[
    source_evidence["recording_id"].isin(suspect_ids)
].drop_duplicates()

if suspect_evidence.empty:
    print("No matching source records recovered from these cache files.")
else:
    print(suspect_evidence.to_string(index=False))

print("\nD. CACHE COVERAGE BY CURRENT STATE LABEL")
recovered_ids = set(source_evidence["recording_id"])
coverage = metadata[["recording_id", "state"]].copy()
coverage["source_recovered"] = coverage["recording_id"].isin(recovered_ids)

print(
    coverage.groupby("state")["source_recovered"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "recovered", "count": "total"})
    .to_string()
)

print("\nNo labels changed. No recordings excluded. No splits created.")

A. RECORDINGS AT THE QUESTIONABLE COORDINATE
recording_id state     scientific_name  latitude  longitude                    source_url
      218546    TX     Spatula discors    35.442   -94.9943 https://xeno-canto.org/218546
      219976    TX Butorides virescens    35.442   -94.9943 https://xeno-canto.org/219976
      218542    TX   Anas carolinensis    35.442   -94.9943 https://xeno-canto.org/218542
      218557    TX  Anas platyrhynchos    35.442   -94.9943 https://xeno-canto.org/218557

B. CACHED XENO-CANTO METADATA
Candidate cache files: 3
  data\raw\xenocanto\xc_metadata_raw_AZ.json
  data\raw\xenocanto\xc_metadata_raw_CA.json
  data\raw\xenocanto\xc_metadata_raw_TX.json

Current cohort IDs recovered: 95
Current cohort IDs without cached evidence: 180

C. ORIGINAL LOCATION EVIDENCE FOR FLAGGED RECORDINGS
recording_id  source_location source_country source_latitude source_longitude     recordist                                source_file
      218546 Sequoyah NWR, OK  United State

In [8]:
# Full-cohort geographic screening
# Downloads reference boundaries only. Does NOT relabel or exclude recordings.

import json
import hashlib
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import pandas as pd
from matplotlib.path import Path as PolygonPath

ENDPOINT = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/"
    "TIGERweb/State_County/MapServer/0/query"
)

params = {
    "where": "STUSAB NOT IN ('AK','HI','PR','VI','GU','MP','AS')",
    "outFields": "STUSAB,NAME",
    "returnGeometry": "true",
    "outSR": "4326",
    "maxAllowableOffset": "0.001",  # degrees; screening simplification
    "f": "geojson",
}

reference_dir = PROJECT_ROOT / "data" / "reference" / "geography"
cache_path = reference_dir / "census_states_screening_v1.geojson"
provenance_path = reference_dir / "census_states_screening_v1_source.json"

if cache_path.exists():
    boundary_bytes = cache_path.read_bytes()
    print("Using cached boundary reference.")
else:
    query_url = ENDPOINT + "?" + urlencode(params)
    print("Downloading Census boundary reference...")
    with urlopen(query_url, timeout=90) as response:
        boundary_bytes = response.read()

boundaries = json.loads(boundary_bytes)

if "error" in boundaries:
    raise RuntimeError(boundaries["error"])

assert boundaries.get("type") == "FeatureCollection"
assert not boundaries.get("exceededTransferLimit", False)

features = boundaries["features"]
available_states = {f["properties"]["STUSAB"] for f in features}
assert {"CA", "AZ", "TX", "OK"}.issubset(available_states)

# Save only a validated reference response; leave existing cache untouched.
if not cache_path.exists():
    reference_dir.mkdir(parents=True, exist_ok=True)
    cache_path.write_bytes(boundary_bytes)
    provenance_path.write_text(
        json.dumps({
            "endpoint": ENDPOINT,
            "parameters": params,
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": hashlib.sha256(boundary_bytes).hexdigest(),
            "purpose": "Geographic screening, not automatic label correction",
        }, indent=2),
        encoding="utf-8",
    )

print("State/DC features loaded:", len(features))
print("Boundary SHA-256:", hashlib.sha256(boundary_bytes).hexdigest())

# Prepare polygons, explicitly preserving exterior rings and holes.
state_polygons = []

for feature in features:
    geometry = feature["geometry"]

    if geometry["type"] == "Polygon":
        polygons = [geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry: {geometry['type']}")

    for polygon in polygons:
        rings = []
        for ring in polygon:
            vertices = np.asarray(ring, dtype=float)[:, :2]
            if not np.array_equal(vertices[0], vertices[-1]):
                vertices = np.vstack([vertices, vertices[0]])
            rings.append(vertices)

        exterior = rings[0]
        state_polygons.append({
            "state": feature["properties"]["STUSAB"],
            "rings": rings,
            "paths": [PolygonPath(ring) for ring in rings],
            "bounds": (
                exterior[:, 0].min(), exterior[:, 1].min(),
                exterior[:, 0].max(), exterior[:, 1].max(),
            ),
        })

def distance_to_ring_km(lon, lat, ring):
    """Local planar distance for a conservative near-boundary screen."""
    scale = np.array([111.32 * np.cos(np.radians(lat)), 111.32])
    xy = (ring - np.array([lon, lat])) * scale

    start = xy[:-1]
    segment = xy[1:] - start
    squared_length = np.sum(segment * segment, axis=1)

    t = np.divide(
        -np.sum(start * segment, axis=1),
        squared_length,
        out=np.zeros_like(squared_length),
        where=squared_length > 0,
    )
    nearest = start + np.clip(t, 0, 1)[:, None] * segment
    return float(np.sqrt(np.sum(nearest * nearest, axis=1)).min())

def screen_coordinate(lon, lat):
    if not np.isfinite([lon, lat]).all():
        return "INVALID", False

    hits = set()
    near_boundary = False

    for polygon in state_polygons:
        xmin, ymin, xmax, ymax = polygon["bounds"]

        # Expanded bounding box is ONLY an acceleration filter.
        # Final membership uses polygons, not rectangles.
        if not (
            xmin - 0.03 <= lon <= xmax + 0.03
            and ymin - 0.03 <= lat <= ymax + 0.03
        ):
            continue

        point = (lon, lat)
        inside = polygon["paths"][0].contains_point(point)
        in_hole = any(
            hole.contains_point(point) for hole in polygon["paths"][1:]
        )

        if inside and not in_hole:
            hits.add(polygon["state"])

        if any(
            distance_to_ring_km(lon, lat, ring) <= 1.0
            for ring in polygon["rings"]
        ):
            near_boundary = True

    return "|".join(sorted(hits)) if hits else "UNMATCHED", near_boundary

# Evaluate repeated coordinates once.
coordinate_results = {}
geo_rows = []

for row in metadata.itertuples(index=False):
    lon = float(row.longitude)
    lat = float(row.latitude)
    key = (lon, lat)

    if key not in coordinate_results:
        coordinate_results[key] = screen_coordinate(lon, lat)

    detected, near = coordinate_results[key]

    if detected in {"INVALID", "UNMATCHED"} or "|" in detected:
        status = "REVIEW_UNRESOLVED"
    elif near:
        status = "REVIEW_NEAR_BOUNDARY"
    elif detected != row.state:
        status = "REVIEW_LABEL_MISMATCH"
    else:
        status = "COORDINATE_AGREES"

    geo_rows.append({
        "recording_id": row.recording_id,
        "original_state": row.state,
        "coordinate_state": detected,
        "near_boundary_1km": near,
        "latitude": lat,
        "longitude": lon,
        "status": status,
    })

geographic_audit = pd.DataFrame(geo_rows)

print("\nORIGINAL LABEL × COORDINATE STATE")
print(pd.crosstab(
    geographic_audit["original_state"],
    geographic_audit["coordinate_state"],
).to_string())

print("\nSCREENING STATUS")
print(geographic_audit["status"].value_counts().to_string())

print("\nRECORDINGS REQUIRING REVIEW")
geographic_review = geographic_audit.loc[
    geographic_audit["status"] != "COORDINATE_AGREES"
]
print(
    geographic_review.to_string(index=False)
    if not geographic_review.empty else "None"
)

print("\nReference cache:", cache_path)
print("Research tables unchanged. No labels corrected or rows excluded.")

Using cached boundary reference.
State/DC features loaded: 49
Boundary SHA-256: d3f6ff87a30cfc2424f028e2d51fc02e655abf2423b354b5826e507ab3f1adfd

ORIGINAL LABEL × COORDINATE STATE
coordinate_state  AR  AZ  CA  LA  NM  NV  OK  OR  TX  UNMATCHED
original_state                                                 
AZ                 0  73   0   0   2   0   0   0   0          0
CA                 0   6  85   0   0   8   0   1   0          0
TX                 1   0   0   6   9   0   7   0  72          5

SCREENING STATUS
status
COORDINATE_AGREES        214
REVIEW_LABEL_MISMATCH     35
REVIEW_NEAR_BOUNDARY      21
REVIEW_UNRESOLVED          5

RECORDINGS REQUIRING REVIEW
recording_id original_state coordinate_state  near_boundary_1km  latitude  longitude                status
      914676             CA               NV              False   41.0393  -119.9806 REVIEW_LABEL_MISMATCH
      140011             CA               AZ              False   32.8688  -114.4516 REVIEW_LABEL_MISMATCH
      738

In [9]:
#  — Create a geographic review ledger with cached source evidence


import json
from datetime import datetime, timezone
import pandas as pd

def first_present(item, keys):
    """Accept alternative source-field names without treating 0 as missing."""
    for key in keys:
        value = item.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return None

# Re-read cached source records using the corrected longitude field mapping.
evidence_rows = []

for path in cache_paths:
    payload = json.loads(path.read_text(encoding="utf-8-sig"))

    for item in source_records(payload):
        recording_id = str(item["id"]).strip()
        if recording_id not in expected_ids:
            continue

        evidence_rows.append({
            "recording_id": recording_id,
            "source_location": first_present(item, ["loc", "location"]),
            "source_country": first_present(item, ["cnt", "country"]),
            "source_latitude": first_present(item, ["lat", "latitude"]),
            "source_longitude": first_present(
                item, ["lon", "lng", "longitude"]
            ),
            "recordist": first_present(item, ["rec", "recordist"]),
            "source_cache": str(path.relative_to(PROJECT_ROOT)),
        })

evidence_columns = [
    "recording_id", "source_location", "source_country",
    "source_latitude", "source_longitude", "recordist", "source_cache",
]

cached_geo_evidence = pd.DataFrame(
    evidence_rows, columns=evidence_columns
).drop_duplicates()

def unique_text(values):
    return " || ".join(sorted({
        str(value).strip()
        for value in values.dropna()
        if str(value).strip()
    }))

# Preserve conflicting/multiple source values rather than choosing one.
if not cached_geo_evidence.empty:
    evidence_summary = (
        cached_geo_evidence.groupby("recording_id", as_index=False)
        .agg({c: unique_text for c in evidence_columns if c != "recording_id"})
    )
else:
    evidence_summary = pd.DataFrame(columns=evidence_columns)

metadata_columns = [
    c for c in [
        "recording_id", "scientific_name", "date", "source_url"
    ]
    if c in metadata.columns
]

geo_review_ledger = (
    geographic_audit
    .merge(
        metadata[metadata_columns],
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        evidence_summary,
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
)

geo_review_ledger["review_priority"] = (
    geo_review_ledger["status"]
    .map({
        "COORDINATE_AGREES": "routine",
        "REVIEW_LABEL_MISMATCH": "manual",
        "REVIEW_NEAR_BOUNDARY": "manual",
        "REVIEW_UNRESOLVED": "manual",
    })
)

# Explicitly blank: no correction is inferred or applied by this cell.
geo_review_ledger["review_decision"] = "pending"
geo_review_ledger["verified_state"] = ""
geo_review_ledger["decision_reason"] = ""
geo_review_ledger["supporting_evidence"] = ""

print("A. SOURCE EVIDENCE COVERAGE")
has_location = (
    geo_review_ledger["source_location"]
    .fillna("").str.strip().ne("")
)
print(pd.crosstab(
    geo_review_ledger["review_priority"],
    has_location.rename("cached_location_available"),
).to_string())

print("\nB. MANUAL REVIEW CASES")
manual_review = geo_review_ledger.loc[
    geo_review_ledger["review_priority"].eq("manual")
].copy()

display_columns = [
    "recording_id", "original_state", "coordinate_state",
    "status", "source_location", "source_country",
    "source_latitude", "source_longitude",
]
print(manual_review[display_columns].fillna("").to_string(index=False))

print("\nC. MANUAL CASES WITHOUT A CACHED LOCATION")
missing_source = manual_review.loc[
    manual_review["source_location"].fillna("").str.strip().eq("")
]
print(
    missing_source[
        ["recording_id", "original_state", "coordinate_state", "source_url"]
    ].to_string(index=False)
)

# New timestamped directory prevents overwriting earlier audit snapshots.
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
audit_dir = (
    PROJECT_ROOT / "data" / "processed" / "supervised"
    / "audit" / f"geographic_review_{run_stamp}"
)
audit_dir.mkdir(parents=True, exist_ok=False)

geo_review_ledger.to_csv(audit_dir / "review_ledger.csv", index=False)
cached_geo_evidence.to_csv(audit_dir / "cached_source_evidence.csv", index=False)

(audit_dir / "provenance.json").write_text(
    json.dumps({
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "boundary_sha256": hashlib.sha256(boundary_bytes).hexdigest(),
        "boundary_reference": str(cache_path.relative_to(PROJECT_ROOT)),
        "recordings": len(geo_review_ledger),
        "manual_review_cases": len(manual_review),
        "decisions_applied": False,
        "note": "Screening evidence only; original research files unchanged.",
    }, indent=2),
    encoding="utf-8",
)

print("\nSaved audit snapshot:", audit_dir)
print("All decisions remain pending. Original files unchanged.")

A. SOURCE EVIDENCE COVERAGE
cached_location_available  False  True 
review_priority                        
manual                        34     27
routine                      146     68

B. MANUAL REVIEW CASES
recording_id original_state coordinate_state                status                                                            source_location source_country source_latitude source_longitude
      914676             CA               NV REVIEW_LABEL_MISMATCH                                                                                                                           
      140011             CA               AZ REVIEW_LABEL_MISMATCH                                                                                                                           
      738058             CA               NV REVIEW_LABEL_MISMATCH                                                                                                                           
      163986             CA 

In [10]:
# Retrieve missing geographic evidence for existing recordings


import os
import json
import time
from datetime import datetime, timezone

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=False)
xc_key = os.getenv("XENO_CANTO_API_KEY")

# Recompute targets from the review ledger rather than an earlier variable.
needs_source = (
    geo_review_ledger["review_priority"].eq("manual")
    & geo_review_ledger["source_location"].fillna("").str.strip().eq("")
)
lookup_ids = sorted(
    geo_review_ledger.loc[needs_source, "recording_id"].astype(str).unique()
)

lookup_dir = (
    PROJECT_ROOT / "data" / "reference"
    / "geography" / "xc_recording_evidence"
)
lookup_dir.mkdir(parents=True, exist_ok=True)

API_URL = "https://xeno-canto.org/api/3/recordings"
lookup_results = []

print("Existing recording IDs requiring metadata:", len(lookup_ids))

with requests.Session() as session:
    for recording_id in lookup_ids:
        destination = lookup_dir / f"XC{recording_id}.json"
        item = None
        retrieval_status = "cached"

        if destination.exists():
            saved = json.loads(destination.read_text(encoding="utf-8"))
            item = saved.get("recording")

            if not isinstance(item, dict) or str(item.get("id")) != recording_id:
                print(f"STOP: cache ID mismatch for XC{recording_id}.")
                break

        else:
            if not xc_key:
                print(
                    "STOP: XENO_CANTO_API_KEY is unavailable. "
                    "Configure it locally; do not paste it into chat."
                )
                break

            try:
                response = session.get(
                    API_URL,
                    params={
                        "query": f"nr:{recording_id}",
                        "key": xc_key,
                        "page": 1,
                    },
                    timeout=(10, 45),
                    allow_redirects=False,
                )
            except requests.RequestException:
                # Avoid displaying exception URLs that could contain the key.
                print(f"STOP: network request failed for XC{recording_id}.")
                break

            if response.status_code != 200:
                print(
                    f"STOP: HTTP {response.status_code} "
                    f"for XC{recording_id}. No automatic retries."
                )
                break

            try:
                payload = response.json()
            except ValueError:
                print(f"STOP: non-JSON response for XC{recording_id}.")
                break

            if not isinstance(payload, dict) or "error" in payload:
                print(f"STOP: API error for XC{recording_id}.")
                break

            matches = [
                record for record in payload.get("recordings", [])
                if str(record.get("id")) == recording_id
            ]

            if len(matches) != 1:
                lookup_results.append({
                    "recording_id": recording_id,
                    "retrieval_status": "not_uniquely_returned",
                })
                print(f"XC{recording_id}: exact record not uniquely returned")
                time.sleep(1)
                continue

            item = matches[0]
            retrieval_status = "retrieved"

            # Cache response evidence, never the API key or authenticated URL.
            destination.write_text(
                json.dumps({
                    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
                    "endpoint": API_URL,
                    "query": f"nr:{recording_id}",
                    "recording": item,
                }, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
            time.sleep(1)

        lookup_results.append({
            "recording_id": recording_id,
            "retrieval_status": retrieval_status,
            "source_location": first_present(item, ["loc", "location"]),
            "source_country": first_present(item, ["cnt", "country"]),
            "source_latitude": first_present(item, ["lat", "latitude"]),
            "source_longitude": first_present(item, ["lon", "lng", "longitude"]),
            "recordist": first_present(item, ["rec", "recordist"]),
            "source_cache": str(destination.relative_to(PROJECT_ROOT)),
        })

        print(f"XC{recording_id}: {retrieval_status}")

new_geo_evidence = pd.DataFrame(lookup_results)

print("\nRETRIEVED EVIDENCE")
if new_geo_evidence.empty:
    print("No records retrieved in this run.")
else:
    columns = [
        c for c in [
            "recording_id", "retrieval_status", "source_location",
            "source_country", "source_latitude", "source_longitude",
        ]
        if c in new_geo_evidence.columns
    ]
    print(new_geo_evidence[columns].fillna("").to_string(index=False))

print("\nEvidence cache:", lookup_dir)
print("Original labels unchanged. No modeling cohort finalized.")

Existing recording IDs requiring metadata: 34
XC120845: cached
XC140011: cached
XC140797: cached
XC163986: cached
XC179040: cached
XC216259: cached
XC216516: cached
XC219976: cached
XC222769: cached
XC228428: cached
XC231530: cached
XC263112: cached
XC317023: cached
XC324293: cached
XC383873: cached
XC423314: cached
XC448002: cached
XC453180: cached
XC486226: cached
XC543757: cached
XC547258: cached
XC551487: cached
XC562113: cached
XC572270: cached
XC626768: cached
XC695151: cached
XC701065: cached
XC738058: cached
XC77507: cached
XC77519: cached
XC820865: cached
XC862339: cached
XC914676: cached
XC917376: cached

RETRIEVED EVIDENCE
recording_id retrieval_status                                                            source_location source_country source_latitude source_longitude
      120845           cached                                         Imperial Dam, Yuma County, Arizona  United States          32.875         -114.457
      140011           cached                       

In [11]:
# Cell 11 — Evidence-based geographic decisions
# Creates a NEW proposed ledger and cohort index.
# Original metadata, features, audio filenames, and previous audits stay intact.

import numpy as np
import pandas as pd
from datetime import datetime, timezone

# Explicit mapping based on the source descriptions reviewed above.
# "MX" is a country code used only for excluded records, not a model class.
reviewed_locations = {
    "CA": "163986 695151 547258",
    "AZ": (
        "140011 562113 453180 121403 120845 551487 "
        "700909 710475 623629 324293 583806 583809 "
        "624546 623628 621691 941132"
    ),
    "TX": "130831 452706 299108",
    "NV": "914676 738058 574653 168805 486226 510630 423314 475180",
    "NM": (
        "626768 636283 543757 495180 233760 448002 "
        "701065 222769 363092 140797 243333"
    ),
    "OK": "228428 218546 820865 219976 218542 231530 218557",
    "LA": "317023 216516 179040 77519 77507 216259",
    "AR": "994154",
    "MX": "656787 862339 917376 383873 263112",
}
held_ids = {"572270"}

source_region = {}
for region, id_string in reviewed_locations.items():
    for recording_id in id_string.split():
        assert recording_id not in source_region, "Duplicate decision ID"
        source_region[recording_id] = region

manual_ids = set(
    geo_review_ledger.loc[
        geo_review_ledger["review_priority"].eq("manual"),
        "recording_id",
    ]
)
assert set(source_region) | held_ids == manual_ids
assert not (set(source_region) & held_ids)

# Combine old cached evidence with newly retrieved evidence.
evidence_columns = [
    "recording_id", "source_location", "source_country",
    "source_latitude", "source_longitude", "source_cache",
]

all_geo_evidence = pd.concat(
    [
        cached_geo_evidence.reindex(columns=evidence_columns),
        new_geo_evidence.reindex(columns=evidence_columns),
    ],
    ignore_index=True,
).drop_duplicates()

# Validate evidence for every manually reviewed recording.
# Conflicting cached values stop the cell rather than being silently discarded.
for recording_id in sorted(manual_ids):
    evidence = all_geo_evidence.loc[
        all_geo_evidence["recording_id"].eq(recording_id)
    ]
    assert not evidence.empty, f"No source evidence: {recording_id}"
    assert evidence["source_location"].fillna("").str.strip().ne("").all(), (
        f"Missing source location: {recording_id}"
    )

    stored = geo_review_ledger.loc[
        geo_review_ledger["recording_id"].eq(recording_id)
    ].iloc[0]

    for source_column, stored_column in [
        ("source_latitude", "latitude"),
        ("source_longitude", "longitude"),
    ]:
        values = pd.to_numeric(evidence[source_column], errors="coerce")
        assert values.notna().all(), (
            f"Missing/non-numeric {source_column}: {recording_id}"
        )
        assert np.isclose(
            values.to_numpy(),
            float(stored[stored_column]),
            atol=1e-5,
            rtol=0,
        ).all(), f"Source/stored coordinate disagreement: {recording_id}"

print("All 61 review cases have source locations and matching stored coordinates.")

def joined_values(values):
    return " || ".join(sorted({
        str(value).strip()
        for value in values.dropna()
        if str(value).strip()
    }))

evidence_summary = (
    all_geo_evidence.groupby("recording_id", as_index=False)
    .agg({
        "source_location": joined_values,
        "source_country": joined_values,
        "source_cache": joined_values,
    })
)

decision_ledger = geographic_audit.merge(
    metadata[["recording_id", "scientific_name", "source_url"]],
    on="recording_id",
    validate="one_to_one",
).merge(
    evidence_summary,
    on="recording_id",
    how="left",
    validate="one_to_one",
)

decisions = []

for row in decision_ledger.itertuples(index=False):
    rid = row.recording_id

    if rid in held_ids:
        action = "HOLD"
        verified_region = ""
        model_state = None
        evidence_level = "source_coordinate_conflict"
        reason = (
            "Source says California side; coordinate screening indicates "
            "Oregon near the border. Further review required."
        )

    elif rid in source_region:
        verified_region = source_region[rid]
        evidence_level = "source_description_and_coordinate_screen"

        if verified_region == "MX":
            assert row.coordinate_state == "UNMATCHED"
        else:
            assert row.coordinate_state == verified_region, (
                f"Source/polygon disagreement: {rid}"
            )

        if verified_region not in {"CA", "AZ", "TX"}:
            action = "EXCLUDE_OUTSIDE_SCOPE"
            model_state = None
            reason = "Source location supports a region outside CA/AZ/TX."
        elif verified_region != row.original_state:
            action = "CORRECT_STATE"
            model_state = verified_region
            reason = "Source description and coordinate screening support AZ."
        else:
            action = "RETAIN"
            model_state = verified_region
            reason = (
                "Boundary-proximate coordinate and source description "
                "support the original state."
            )

    else:
        assert row.status == "COORDINATE_AGREES"
        action = "RETAIN"
        verified_region = row.coordinate_state
        model_state = row.original_state
        evidence_level = "coordinate_screen_only"
        reason = (
            "Coordinate agrees away from the review buffer; "
            "not independent verification of the source coordinate."
        )

    decisions.append({
        "recording_id": rid,
        "action": action,
        "evidence_region": verified_region,
        "model_state": model_state,
        "evidence_level": evidence_level,
        "decision_reason": reason,
    })

decision_ledger = decision_ledger.merge(
    pd.DataFrame(decisions),
    on="recording_id",
    validate="one_to_one",
)
decision_ledger["protocol_status"] = "proposed_geographic_correction_v1"

# This is an INDEX, not a modified feature table.
provisional_cohort_index = decision_ledger.loc[
    decision_ledger["action"].isin(["RETAIN", "CORRECT_STATE"]),
    ["recording_id", "original_state", "model_state",
     "scientific_name", "action", "evidence_level"],
].copy()

counts = decision_ledger["action"].value_counts()
assert counts.to_dict() == {
    "RETAIN": 230,
    "EXCLUDE_OUTSIDE_SCOPE": 38,
    "CORRECT_STATE": 6,
    "HOLD": 1,
}
assert len(provisional_cohort_index) == 236
assert provisional_cohort_index["recording_id"].is_unique

print("\nDECISIONS")
print(counts.to_string())

print("\nPROVISIONAL MODELING COUNTS")
print(provisional_cohort_index["model_state"].value_counts().to_string())

print("\nCORRECTED LABELS")
print(
    provisional_cohort_index.loc[
        provisional_cohort_index["action"].eq("CORRECT_STATE"),
        ["recording_id", "original_state", "model_state", "scientific_name"],
    ].to_string(index=False)
)

print("\nEXCLUDED REGIONS")
print(
    decision_ledger.loc[
        decision_ledger["action"].eq("EXCLUDE_OUTSIDE_SCOPE"),
        "evidence_region",
    ].value_counts().to_string()
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
decision_dir = (
    PROJECT_ROOT / "data" / "processed" / "supervised"
    / "audit" / f"geographic_decisions_{stamp}"
)
decision_dir.mkdir(parents=True, exist_ok=False)

decision_ledger.to_csv(decision_dir / "decision_ledger.csv", index=False)
provisional_cohort_index.to_csv(
    decision_dir / "provisional_cohort_index.csv", index=False
)
all_geo_evidence.to_csv(decision_dir / "source_evidence.csv", index=False)

print("\nSaved:", decision_dir)
print("Original datasets unchanged. Cohort remains provisional.")

All 61 review cases have source locations and matching stored coordinates.

DECISIONS
action
RETAIN                   230
EXCLUDE_OUTSIDE_SCOPE     38
CORRECT_STATE              6
HOLD                       1

PROVISIONAL MODELING COUNTS
model_state
CA    85
AZ    79
TX    72

CORRECTED LABELS
recording_id original_state model_state     scientific_name
      140011             CA          AZ   Gallinula galeata
      562113             CA          AZ      Columbina inca
      453180             CA          AZ Antigone canadensis
      121403             CA          AZ  Anser caerulescens
      120845             CA          AZ     Rallus limicola
      551487             CA          AZ      Columbina inca

EXCLUDED REGIONS
evidence_region
NM    11
NV     8
OK     7
LA     6
MX     5
AR     1

Saved: C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\supervised\audit\geographic_decisions_20261101T073749454708Z
Original datasets unchanged. Cohort remains provisi

In [12]:
# Cell 12 — Corrected-cohort support and quality
# Read-only; no splitting or further exclusions.

import pandas as pd

cohort = provisional_cohort_index.merge(
    metadata.drop(columns=["state"]),
    on=["recording_id", "scientific_name"],
    how="left",
    validate="one_to_one",
).merge(
    recordings[[
        "recording_id", "n_windows", "flag_bandlimit_incompatible"
    ]],
    on="recording_id",
    how="left",
    validate="one_to_one",
)

assert len(cohort) == 236
assert cohort["recording_id"].is_unique
assert cohort["model_state"].notna().all()

print("A. SPECIES SUPPORT")
species_counts = pd.crosstab(
    cohort["scientific_name"], cohort["model_state"]
).reindex(columns=["CA", "AZ", "TX"], fill_value=0)

states_present = species_counts.gt(0).sum(axis=1)
totals = species_counts.sum(axis=1)

print("Species:", len(species_counts))
print("Singleton species:", int(totals.eq(1).sum()))
print("Species by number of states represented:")
print(states_present.value_counts().sort_index().to_string())

print("\nPairwise replication:")
for a, b in [("CA", "AZ"), ("CA", "TX"), ("AZ", "TX")]:
    shared = species_counts[a].gt(0) & species_counts[b].gt(0)
    replicated = species_counts[a].ge(2) & species_counts[b].ge(2)
    print(f"{a}–{b}: shared={shared.sum()}, ≥2 in each={replicated.sum()}")

exclusive_species = set(species_counts.index[states_present.eq(1)])
cohort["sample_exclusive_species"] = (
    cohort["scientific_name"].isin(exclusive_species)
)

print("\nSample-exclusive species recordings by state:")
print(
    cohort.groupby("model_state")["sample_exclusive_species"]
    .agg(["sum", "count", "mean"]).round(3).to_string()
)

print("\nB. GEOGRAPHIC SUPPORT")
cohort["coordinate_key"] = list(zip(
    pd.to_numeric(cohort["latitude"], errors="raise"),
    pd.to_numeric(cohort["longitude"], errors="raise"),
))

location_counts = (
    cohort.groupby(["model_state", "coordinate_key"])
    .size().rename("recordings").reset_index()
)
print(
    location_counts.groupby("model_state")["recordings"]
    .agg(
        coordinate_groups="size",
        largest_group="max",
        median_group="median",
    ).to_string()
)
print("\nLargest coordinate groups:")
print(
    location_counts.sort_values("recordings", ascending=False)
    .head(10).to_string(index=False)
)

print("\nC. QUALITY SUPPORT")
flag_columns = [
    "flag_short_duration", "flag_low_amplitude",
    "flag_clipping", "flag_any", "flag_bandlimit_incompatible",
]

flag_values = {}
for column in flag_columns:
    normalized = cohort[column].astype("string").str.strip().str.lower()
    assert normalized.isin(
        ["true", "false", "1", "0", "1.0", "0.0"]
    ).all(), f"Unexpected flag values: {column}"
    flag_values[column] = normalized.isin(["true", "1", "1.0"])

quality_flags = pd.DataFrame(flag_values, index=cohort.index)
print(
    quality_flags.groupby(cohort["model_state"]).sum().to_string()
)

print("\nDuration and windows:")
cohort["duration_seconds"] = pd.to_numeric(
    cohort["duration_seconds"], errors="raise"
)
print(
    cohort.groupby("model_state").agg(
        recordings=("recording_id", "size"),
        median_duration=("duration_seconds", "median"),
        minimum_duration=("duration_seconds", "min"),
        median_windows=("n_windows", "median"),
    ).round(3).to_string()
)

inspect = (
    quality_flags["flag_short_duration"]
    | quality_flags["flag_bandlimit_incompatible"]
)
print("\nShort-duration or bandwidth-flagged recordings:")
print(
    cohort.loc[inspect, [
        "recording_id", "model_state", "scientific_name",
        "duration_seconds", "n_windows",
        "flag_short_duration", "flag_bandlimit_incompatible",
    ]].to_string(index=False)
)

print("\nNo new exclusions or splits. Corrected cohort is in memory as `cohort`.")

A. SPECIES SUPPORT
Species: 134
Singleton species: 79
Species by number of states represented:
1    98
2    30
3     6

Pairwise replication:
CA–AZ: shared=29, ≥2 in each=14
CA–TX: shared=9, ≥2 in each=0
AZ–TX: shared=10, ≥2 in each=0

Sample-exclusive species recordings by state:
             sum  count   mean
model_state                   
AZ            24     79  0.304
CA            34     85  0.400
TX            59     72  0.819

B. GEOGRAPHIC SUPPORT
             coordinate_groups  largest_group  median_group
model_state                                                
AZ                          43             16           1.0
CA                          70              3           1.0
TX                          57              4           1.0

Largest coordinate groups:
model_state       coordinate_key  recordings
         AZ  (31.906, -109.1543)          16
         AZ    (31.34, -109.273)           8
         TX  (26.1253, -97.9532)           4
         AZ (31.7244, -110.8805)

In [13]:
# Cell 13 — Freeze proposed eligibility policy and membership
# Writes new audit artifacts only. No model fitting or data overwrites.

import json
import hashlib
from datetime import datetime, timezone
import pandas as pd

assert len(cohort) == 236
assert cohort["recording_id"].is_unique
assert quality_flags.index.equals(cohort.index)

eligibility = cohort[[
    "recording_id", "original_state", "model_state", "scientific_name"
]].copy()

eligibility["primary_eligible"] = True
eligibility["short_duration_flag"] = (
    quality_flags["flag_short_duration"].to_numpy()
)
eligibility["bandwidth_flag"] = (
    quality_flags["flag_bandlimit_incompatible"].to_numpy()
)
eligibility["sensitivity_eligible"] = ~(
    eligibility["short_duration_flag"] | eligibility["bandwidth_flag"]
)

# Explicit reasons: no exclusion based on species identity or model scores.
eligibility["sensitivity_exclusion_reason"] = [
    ";".join(reason for flag, reason in [
        (short, "existing_short_duration_flag"),
        (band, "existing_bandlimit_incompatible_flag"),
    ] if flag)
    for short, band in zip(
        eligibility["short_duration_flag"],
        eligibility["bandwidth_flag"],
    )
]

assert eligibility["sensitivity_eligible"].sum() == 230
assert len(mean_features) == 50
assert len(set(mean_features)) == 50

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

policy = {
    "version": "part3_eligibility_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "primary_question": "Original Q7",
    "target_column": "model_state",
    "classes": ["CA", "AZ", "TX"],
    "observational_unit": "recording",
    "primary_cohort_n": 236,
    "sensitivity_cohort_n": 230,
    "primary_policy": (
        "Geographically eligible recordings; retain quality flags "
        "for diagnostics rather than automatic exclusions."
    ),
    "sensitivity_policy": (
        "Exclude existing short-duration or bandwidth-incompatible flags. "
        "This is a sensitivity analysis, not a validated biological threshold."
    ),
    "feature_allowlist": list(mean_features),
    "grouping_requirement": (
        "Known shared encounters and exact-coordinate groups must not "
        "cross the primary train/test boundary. Coordinate grouping is "
        "a proxy and does not guarantee recordist or nearby-site separation."
    ),
    "limitations": [
        "Selected focal-recording collection, not standardized soundscapes.",
        "Species composition and regional labels remain confounded.",
        "Quality and recording duration vary by region.",
        "One geographically disputed recording remains held out of the cohort.",
    ],
    "input_hashes": {
        label: file_sha256(PROJECT_ROOT / relative)
        for label, relative in REQUIRED_FILES.items()
    },
    "models_fitted": False,
    "split_created": False,
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
eligibility_dir = (
    PROJECT_ROOT / "data" / "processed" / "supervised"
    / "audit" / f"eligibility_{stamp}"
)
eligibility_dir.mkdir(parents=True, exist_ok=False)

eligibility.to_csv(eligibility_dir / "cohort_membership.csv", index=False)
policy_path = eligibility_dir / "eligibility_policy.json"
policy_path.write_text(
    json.dumps(policy, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("PRIMARY COHORT")
print(eligibility.groupby("model_state").size().to_string())

print("\nSENSITIVITY SUBSET")
print(
    eligibility.loc[eligibility["sensitivity_eligible"]]
    .groupby("model_state").size().to_string()
)

print("\nSENSITIVITY EXCLUSIONS")
print(
    eligibility.loc[
        ~eligibility["sensitivity_eligible"],
        ["recording_id", "model_state", "sensitivity_exclusion_reason"],
    ].to_string(index=False)
)

print("\nSaved:", eligibility_dir)
print("Policy SHA-256:", file_sha256(policy_path))
print("No models fitted. No splits created. Original files unchanged.")

PRIMARY COHORT
model_state
AZ    79
CA    85
TX    72

SENSITIVITY SUBSET
model_state
AZ    78
CA    83
TX    69

SENSITIVITY EXCLUSIONS
recording_id model_state         sensitivity_exclusion_reason
      691650          CA         existing_short_duration_flag
      779907          CA         existing_short_duration_flag
       75505          AZ         existing_short_duration_flag
      288791          TX         existing_short_duration_flag
      123297          TX existing_bandlimit_incompatible_flag
      845953          TX         existing_short_duration_flag

Saved: C:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\supervised\audit\eligibility_20261101T073749614594Z
Policy SHA-256: 9404cc48e3821b46143618a30b84c480cf95a6b0407f5a5811199033564f9c19
No models fitted. No splits created. Original files unchanged.
